# Chapter 6 — Comprehensions and Generators

Source: *Effective Python*, 3rd Edition (Brett Slatkin), Chapter 6.

---


### Item 40 — Use Comprehensions Instead of `map` and `filter`

A list comprehension (`[x*2 for x in items if x > 0]`) does what `map()`+`filter()` do combined, without needing a separate `lambda` for each step and without the visual indirection of nested function calls — `map`/`filter` also return iterators in Python 3, so you'd need to wrap in `list()` anyway to see the result, whereas a comprehension already gives you a concrete list directly.


In [1]:
numbers = [-4, -2, 0, 3, 7, -1, 9]

# map + filter: needs two separate lambdas, plus list() to materialize the iterator
doubled_positive_mf = list(map(lambda x: x * 2, filter(lambda x: x > 0, numbers)))
print(doubled_positive_mf)

# Comprehension: same result, one readable line, no lambdas needed
doubled_positive = [x * 2 for x in numbers if x > 0]
print(doubled_positive)


[6, 14, 18]
[6, 14, 18]


### Item 41 — Avoid More Than Two Control Subexpressions in Comprehensions

Comprehensions can chain multiple `for` clauses and `if` conditions, but readability collapses fast past two control subexpressions total (loops + conditions combined) — a comprehension with 3 nested `for`s and 2 `if`s is a genuine puzzle to read back. Past that complexity, a normal multi-line `for` loop (or a helper function) is clearer, even though it's longer.


In [2]:
matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]

# Two control subexpressions (one for, one if): still reasonably clear
evens_flat = [x for row in matrix for x in row if x % 2 == 0]
print(evens_flat)

# Past that -- e.g. two fors AND two ifs -- readability drops fast (illustrative, not recommended):
overly_complex = [x for row in matrix if len(row) == 3 for x in row if x % 2 == 0 if x > 2]
print('technically works, but hard to read at a glance:', overly_complex)

# Clearer alternative once complexity grows: a plain loop
result = []
for row in matrix:
    if len(row) != 3:
        continue
    for x in row:
        if x % 2 == 0 and x > 2:
            result.append(x)
print('same result, easier to follow:', result)


[2, 4, 6, 8]
technically works, but hard to read at a glance: [4, 6, 8]
same result, easier to follow: [4, 6, 8]


### Item 42 — Reduce Repetition in Comprehensions with Assignment Expressions

If a comprehension needs to use the same computed value twice (once to filter, once in the output expression), naively it gets computed twice. The walrus operator lets you compute it once, inline, and reuse the bound name in both places within the same comprehension.


In [3]:
import math

numbers = [16, 25, 30, 9, 41]

# Without walrus: sqrt is computed TWICE for every qualifying item (once to check, once to use)
result_repeated = [math.sqrt(n) for n in numbers if math.sqrt(n) == int(math.sqrt(n))]
print(result_repeated)

# With walrus: computed once, reused via the bound name `root`
result_once = [root for n in numbers if (root := math.sqrt(n)) == int(root)]
print(result_once)


[4.0, 5.0, 3.0]
[4.0, 5.0, 3.0]


### Item 43 — Consider Generators Instead of Returning Lists

A function that builds and returns a full list forces every result to exist in memory at once, even if the caller only needed to loop over them one at a time. Making it a generator (using `yield` instead of building/returning a list) produces items lazily, one at a time, using a constant, small amount of memory regardless of how many results there are in total.


In [4]:
def find_positions_list(text, target):
    positions = []
    for i, char in enumerate(text):
        if char == target:
            positions.append(i)
    return positions   # entire list must exist in memory before the caller sees anything

def find_positions_gen(text, target):
    for i, char in enumerate(text):
        if char == target:
            yield i   # produces ONE position at a time, on demand

text = 'the quick brown fox jumps over the lazy dog'
print(find_positions_list(text, 'o'))
print(list(find_positions_gen(text, 'o')))   # same result, but computed lazily under the hood

# The generator version lets you stop early without wasting work on the rest
gen = find_positions_gen(text, 'o')
print('first match only:', next(gen))


[12, 17, 26, 41]
[12, 17, 26, 41]
first match only: 12


### Item 44 — Consider Generator Expressions for Large List Comprehensions

A list comprehension `[x**2 for x in range(10_000_000)]` builds the *entire* list in memory immediately. Wrapping the same expression in parentheses instead of brackets — `(x**2 for x in range(10_000_000))` — makes it a **generator expression**: values are computed lazily, one at a time, as something actually asks for the next one, which is essential once the input is large or unbounded.


In [5]:
import sys

list_comp = [x**2 for x in range(100_000)]        # fully built in memory right now
gen_expr = (x**2 for x in range(100_000))          # nothing computed yet -- lazy

print('list comprehension size:', sys.getsizeof(list_comp), 'bytes')
print('generator expression size:', sys.getsizeof(gen_expr), 'bytes')   # tiny, regardless of the 100k range

total = sum(x**2 for x in range(100_000))   # generator expr feeds sum() lazily, one value at a time
print('sum computed via generator:', total)


list comprehension size: 800984 bytes
generator expression size: 104 bytes
sum computed via generator: 333328333350000


### Item 45 — Compose Multiple Generators with `yield from`

When one generator needs to delegate to another (yield everything the inner generator produces, then continue), `yield from inner_generator` does this in one line — equivalent to, but clearer and slightly faster than, manually looping over the inner generator and yielding each item yourself.


In [6]:
def count_up(n):
    for i in range(1, n + 1):
        yield i

def count_down(n):
    for i in range(n, 0, -1):
        yield i

def up_then_down(n):
    yield from count_up(n)     # delegates entirely to count_up's items
    yield from count_down(n)   # then delegates to count_down's items

print(list(up_then_down(4)))

# Without yield from, you'd have to manually re-loop and re-yield:
def up_then_down_manual(n):
    for i in count_up(n):
        yield i
    for i in count_down(n):
        yield i

print(list(up_then_down_manual(4)))   # same result, more code to say the same thing


[1, 2, 3, 4, 4, 3, 2, 1]
[1, 2, 3, 4, 4, 3, 2, 1]


### Item 46 — Pass Iterators into Generators as Arguments Instead of Calling the `send` Method

A generator's `.send(value)` method can inject a value back into a paused generator (received as the result of the `yield` expression) — but this makes the generator's control flow hard to follow, since values now arrive from two different directions. The book's advice: prefer passing an **iterator as a normal argument** to the generator function instead, so the generator just pulls values from it with `next()` in the usual, easy-to-trace direction.


In [7]:
# send()-based approach: values flow INTO the generator via .send() -- harder to trace
def averager_send():
    total, count = 0, 0
    average = None
    while True:
        value = yield average    # yield produces `average`, .send() delivers the next `value`
        total += value
        count += 1
        average = total / count

gen = averager_send()
next(gen)               # prime it (advances to the first yield)
print(gen.send(10))
print(gen.send(20))

# Preferred: pass an iterator IN as a normal argument -- one clear direction of data flow
def averager_from_iterator(values):
    total, count = 0, 0
    for value in values:
        total += value
        count += 1
        yield total / count

for avg in averager_from_iterator([10, 20, 30]):
    print(avg)


10.0
15.0
10.0
15.0
20.0


### Item 47 — Manage Iterative State Transitions with a Class Instead of the Generator `throw` Method

A generator's `.throw(exception)` method raises an exception *inside* the paused generator, at the point of its current `yield` — a rarely-needed, hard-to-follow mechanism for injecting error states. The book's advice: if a generator's logic needs to change behavior based on external events/errors, model that as an explicit state machine using a normal class with methods, which is far easier to reason about than exceptions being thrown into the middle of paused generator code.


In [8]:
# throw()-based approach: an exception is injected INTO a paused generator -- hard to trace
def resettable_counter_throw():
    count = 0
    while True:
        try:
            yield count
            count += 1
        except ValueError:
            count = 0   # reset triggered by an exception thrown from OUTSIDE

gen = resettable_counter_throw()
print(next(gen), next(gen), next(gen))
print(gen.throw(ValueError))   # injects an exception into the generator's paused state
print(next(gen))

# Preferred: an explicit class-based state machine -- state transitions are just normal method calls
class ResettableCounter:
    def __init__(self):
        self.count = 0
    def next_value(self):
        value = self.count
        self.count += 1
        return value
    def reset(self):
        self.count = 0

counter = ResettableCounter()
print(counter.next_value(), counter.next_value(), counter.next_value())
counter.reset()   # a plain, traceable method call -- no exception machinery involved
print(counter.next_value())


0 1 2
0
1
0 1 2
0
